# MASH (Multivariate Adaptive Shrinkage)
This is new statistical methods for analyzing genomic data sets that measure many effects in many conditions (for example, gene expression changes under many treatments). These new methods improve on existing methods by allowing for arbitrary correlations in effect sizes among conditions. This flexible approach increases power, improves effect estimates and allows for more quantitative assessments of effect-size heterogeneity compared to simple shared or condition-specific assessments. [methods paper](https://www.nature.com/articles/s41588-018-0268-8)

MASH to improve cross-population transcriptome prediction, [preprint](https://pubmed.ncbi.nlm.nih.gov/36798214/)

# mash_rand_null_sample() Function in mash_wrapper.R of pecotmr

In [ ]:
mash_rand_null_sample <- function(dat, n_random, n_null, exclude_condition, seed = NULL) {
  # Function to extract one data set
  extract_one_data <- function(dat, n_random, n_null) {
    if (is.null(dat)) {
      return(NULL)
    }
    abs_z <- abs(dat$z)
    sample_idx <- 1:nrow(abs_z)
    random_idx <- sample(sample_idx, min(n_random, length(sample_idx)), replace = FALSE)
    random <- list(z = dat$z[random_idx, , drop = FALSE])
    null.id <- which(apply(abs_z, 1, max) < 2)
    if (length(null.id) == 0) {
      warning(paste("no variants are included in the null dataset because abs_z > 2 for all variants in", dat$region))
      null <- list()
    } else {
      if (length(null.id) < ncol(dat$z)) {
        warning(paste("not enough null data to estimate null correlation in", dat$region))
        null <- list()
      } else {
        null_idx <- sample(null.id, min(n_null, length(null.id)), replace = FALSE)
        null <- list(z = dat$z[null_idx, , drop = FALSE])
      }
    }
    dat <- list(random = random, null = null)
    return(dat)
  }

  if (!is.null(seed)) {
    set.seed(seed)
  }
  if (length(exclude_condition) > 0) {
    if (all(exclude_condition %in% colnames(dat$z))) {
      dat$z <- dat$z[, -exclude_condition, drop = FALSE]
    } else {
      # Handle the case where exclude_condition names do not exist in column
      # names of dat This could be an error
      stop(paste("Error: exclude_condition are not present in", dat$region))
    }
  }
  result <- extract_one_data(dat, n_random, n_null)
  return(result)
}

The function is designed to work with genetic data for a statistical method called MASH (Multivariate Adaptive Shrinkage).
                                                  
* It creates two subsets of data: a random sample and a "null" sample (where all absolute z-scores are < 2).
* It includes error handling for various edge cases, such as insufficient null data.
* The function allows for exclusion of certain conditions (columns) from the analysis.
* It uses random sampling, making the seed parameter important for reproducibility.

This check ensures that the subsequent analysis (presumably involving MASH or a similar method) has sufficient data to make reliable inferences about the null distribution of effects across conditions.

## Ability to Calculate Null Correlation:
If you can calculate the null correlation, it doesn't necessarily mean your entire dataset follows a null distribution. Rather, it means you have enough data points that appear to come from a null distribution to estimate its properties.

## What the Null Sample Represents:
The null sample in this function consists of data points where the absolute z-scores are less than 2 across all conditions. These are considered "null-like" because they show relatively small effects that could plausibly arise from random chance rather than true associations.

## Implications:

Having enough null-like data points to estimate the null correlation is a good sign. It suggests your dataset includes a substantial subset of observations that behave as expected under the null hypothesis.
However, this doesn't mean your entire dataset follows a null distribution. In fact, in most studies, you hope to have both null-like observations and observations showing real effects.
Purpose of Null Correlation:

The null correlation is used as a baseline or reference point in methods like MASH (Multivariate Adaptive Shrinkage).
It helps to model the background noise and correlation structure that exists even when there's no true effect.
This information is then used to better identify and estimate real effects in the data.
## Interpretation:

If you can calculate the null correlation, it suggests your data includes enough null-like observations to estimate background patterns.
This is generally a good thing for analyses that contrast potential effects against a null background.
It doesn't mean your entire dataset is null, but rather that you have a good reference point for identifying non-null effects.


**In summary, being able to calculate the null correlation indicates you have sufficient null-like data to establish a baseline, which is crucial for methods that aim to distinguish true signals from noise in complex, multi-condition datasets. It's a positive aspect for your analysis, but it doesn't imply the absence of real effects in your data.**